# Indexing, reshape, join, and broadcasting

How to select values, change layout, glue tensors together, and add tensors of different shapes.


In [ ]:
import torch


## 1. Slicing


In [ ]:
y_1d = torch.arange(10)
print("slice [2:5]:", y_1d[2:5])
print("every other element:", y_1d[::2])
print("flipped:", y_1d.flip(dims=[0]))


## 2. Boolean indexing


In [ ]:
data = torch.tensor([[1, 2], [3, 4], [5, 6]])
mask = data > 3
print("mask:\n", mask)
print("values where mask is True:", data[mask])

data[data <= 3] = 0
print("after zeroing small values:\n", data)
print("rows whose first column > 2:\n", data[data[:, 0] > 2, :])


## 3. Integer-array indexing


In [ ]:
x = torch.arange(10, 20)
indices = torch.tensor([0, 4, 2, 2])
print("selected:", x[indices])

new_x = torch.arange(12).reshape(3, 4)
row_idx = torch.tensor([0, 1, 2])
col_idx = torch.tensor([1, 3, 0])
print("pairs (0,1), (1,3), (2,0):", new_x[row_idx, col_idx])


## 4. `view` vs `reshape`

- `view` only works if the tensor is **contiguous** in memory.
- Transpose (`t()` / `permute`) often breaks contiguity.
- `reshape` will copy if it must. `contiguous()` makes a contiguous copy so `view` can succeed.


In [ ]:
x = torch.arange(16)
y = x.view(4, 4)
z = y.t()
print("z contiguous?", z.is_contiguous())

a = torch.arange(12).view(3, 4)
b = a.t()
print("b contiguous?", b.is_contiguous())

try:
    c = b.view(6, 2)
except RuntimeError as e:
    print("view failed as expected:")
    print(e)

print("reshape works:", b.reshape(6, 2).shape)
print("after contiguous, view works:", b.contiguous().view(6, 2).shape)


## 5. `permute`

Typical image layout in PyTorch is `N × C × H × W`. Matplotlib wants `H × W × C`.


In [ ]:
x = torch.randn(3, 32, 32)  # C, H, W
hwc = x.permute(1, 2, 0)
print("CHW -> HWC:", x.shape, "->", hwc.shape)
print("permute is contiguous?", hwc.is_contiguous())

flat = x.contiguous().view(-1)
print("flattened length:", flat.numel())


## 6. Join: `cat` vs `stack`

- `cat` glues along an **existing** dimension (shapes must match on the other dims).
- `stack` creates a **new** dimension.


In [ ]:
a = torch.arange(1, 7).reshape(2, 3)
b = torch.arange(7, 13).reshape(2, 3)
extra_rows = torch.randn(4, 3)

print("cat along dim=0 (rows):", torch.cat((a, extra_rows), dim=0).shape)
print("stack dim=0:", torch.stack((a, b), dim=0).shape)
print("stack dim=1:", torch.stack((a, b), dim=1).shape)


## 7. Split: `split` vs `chunk`

- `split(size)`: each chunk has `size` along that dim (last chunk may be smaller). We can also pass a list of sizes.
- `chunk(n)`: cut into `n` pieces as evenly as possible.


In [ ]:
tensor_g = torch.arange(12).reshape(6, 2)

print("equal split of 2 rows:")
for i, chunk in enumerate(torch.split(tensor_g, 2, dim=0)):
    print(i, chunk.shape)

print("unequal split [1, 2, 3]:")
for i, chunk in enumerate(torch.split(tensor_g, [1, 2, 3], dim=0)):
    print(i, chunk.shape)

tensor_h = torch.arange(10).reshape(5, 2)
print("chunk into 3 along dim=0:")
for i, chunk in enumerate(torch.chunk(tensor_h, 3, dim=0)):
    print(i, chunk.shape)


## 8. Broadcasting

PyTorch aligns shapes **from the right**. A `(3,)` vector added to a `(3, 3)` matrix is treated as one value per **column**. Reshape to `(3, 1)` to add per **row**.


In [ ]:
matrix = torch.arange(1, 10).reshape(3, 3)
row_vec = torch.arange(10, 31, 10)  # [10, 20, 30]

print("matrix:\n", matrix)
print("matrix + row_vec (broadcast on columns):\n", matrix + row_vec)
print("matrix + row_vec.reshape(3, 1) (broadcast on rows):\n", matrix + row_vec.reshape(3, 1))
